In [0]:
import pandas as pd
 
data = [
    (1, '2024-01-01', '2024-01-05', '2024-01-06', 'North', 'FastX', 'Delhi_WH', 2000, 1, 150),
    (2, '2024-01-02', '2024-01-06', '2024-01-06', 'South', 'QuickShip', 'Bangalore_WH', 1500, 0, 0),
    (3, '2024-01-03', '2024-01-07', '2024-01-10', 'East', 'DeliverIT', 'Kolkata_WH', 3000, 3, 600),
    (4, '2024-01-04', '2024-01-08', '2024-01-08', 'West', 'ShipNow', 'Mumbai_WH', 2500, 0, 0),
    (5, '2024-01-05', '2024-01-09', '2024-01-11', 'North', 'FastX', 'Delhi_WH', 4000, 2, 300),
    (6, '2024-01-06', '2024-01-10', '2024-01-12', 'South', 'QuickShip', 'Bangalore_WH', 1800, 2, 200),
    (7, '2024-01-07', '2024-01-11', '2024-01-11', 'East', 'DeliverIT', 'Kolkata_WH', 2200, 0, 0),
    (8, '2024-01-08', '2024-01-12', '2024-01-14', 'West', 'ShipNow', 'Mumbai_WH', 3500, 2, 400)
]
 
columns = [
    'order_id', 'order_date', 'expected_delivery_date', 'actual_delivery_date',
    'delivery_region', 'courier_partner', 'warehouse_location',
    'order_value', 'delay_days', 'penalty'
]
 
df = pd.DataFrame(data, columns=columns)
 
print(df)

df['order_date'] = pd.to_datetime(df['order_date'])
df['expected_delivery_date'] = pd.to_datetime(df['expected_delivery_date'])
df['actual_delivery_date'] = pd.to_datetime(df['actual_delivery_date'])
print(df)

   order_id  order_date  ... delay_days penalty
0         1  2024-01-01  ...          1     150
1         2  2024-01-02  ...          0       0
2         3  2024-01-03  ...          3     600
3         4  2024-01-04  ...          0       0
4         5  2024-01-05  ...          2     300
5         6  2024-01-06  ...          2     200
6         7  2024-01-07  ...          0       0
7         8  2024-01-08  ...          2     400

[8 rows x 10 columns]
   order_id order_date expected_delivery_date  ... order_value delay_days penalty
0         1 2024-01-01             2024-01-05  ...        2000          1     150
1         2 2024-01-02             2024-01-06  ...        1500          0       0
2         3 2024-01-03             2024-01-07  ...        3000          3     600
3         4 2024-01-04             2024-01-08  ...        2500          0       0
4         5 2024-01-05             2024-01-09  ...        4000          2     300
5         6 2024-01-06             2024-01-10  ...   

In [0]:
#Problem1-Which region has highest delay rate?
df['is_delayed'] = df['delay_days'] > 0
print(df)

   order_id order_date expected_delivery_date  ... delay_days penalty is_delayed
0         1 2024-01-01             2024-01-05  ...          1     150       True
1         2 2024-01-02             2024-01-06  ...          0       0      False
2         3 2024-01-03             2024-01-07  ...          3     600       True
3         4 2024-01-04             2024-01-08  ...          0       0      False
4         5 2024-01-05             2024-01-09  ...          2     300       True
5         6 2024-01-06             2024-01-10  ...          2     200       True
6         7 2024-01-07             2024-01-11  ...          0       0      False
7         8 2024-01-08             2024-01-12  ...          2     400       True

[8 rows x 11 columns]


In [0]:

region_delay_rate = (
    df.groupby('delivery_region')
      .agg(
          total_orders=('order_id', 'count'),
          delayed_orders=('is_delayed', 'sum')
      )
)

region_delay_rate['delay_rate_%'] = (
    region_delay_rate['delayed_orders'] /
    region_delay_rate['total_orders']
) * 100
print(region_delay_rate)

                 total_orders  delayed_orders  delay_rate_%
delivery_region                                            
East                        2               1          50.0
North                       2               2         100.0
South                       2               1          50.0
West                        2               1          50.0


In [0]:
region_delay_rate=region_delay_rate.reset_index()
region_delay_rate.to_csv("region_delay_rate1.csv",index=False)

In [0]:
#Problem2-Which courier partner is causing most delays?
courier_delay_rate = (
    df.groupby('courier_partner')
             .agg(
                 total_orders=('order_id', 'count'),
                 delayed_orders=('is_delayed', 'sum')
             )
)

courier_delay_rate['delay_rate_%'] = (
    courier_delay_rate['delayed_orders'] /
    courier_delay_rate['total_orders']
) * 100
print(courier_delay_rate)

                 total_orders  delayed_orders  delay_rate_%
courier_partner                                            
DeliverIT                   2               1          50.0
FastX                       2               2         100.0
QuickShip                   2               1          50.0
ShipNow                     2               1          50.0


In [0]:
courier_delay_rate=courier_delay_rate.reset_index()
courier_delay_rate.to_csv("courier_delay_rate.csv",index=False)

In [0]:
#Problem3-How much penalty loss is happening?

total_penalty_loss = df['penalty'].sum()
print("Total penalty loss:", total_penalty_loss)


Total penalty loss: 1650


In [0]:

penalty_by_region = df.groupby('delivery_region')['penalty'].sum()
print(penalty_by_region)


delivery_region
East     600
North    450
South    200
West     400
Name: penalty, dtype: int64


In [0]:
penalty_by_courier = df.groupby('courier_partner')['penalty'].sum()
print(penalty_by_courier)

courier_partner
DeliverIT    600
FastX        450
QuickShip    200
ShipNow      400
Name: penalty, dtype: int64


In [0]:
delayed_penalty = df[df['is_delayed']]['penalty'].sum()
print("Penalty due to delayed orders:", delayed_penalty)

Penalty due to delayed orders: 1650


In [0]:
penalty_table=df.groupby(['delivery_region','courier_partner'])['penalty'].sum().reset_index(name='total_penalty_by_region')
print(penalty_table)
penalty_table.to_csv("penalty_table.csv",index=False)

  delivery_region courier_partner  total_penalty_by_region
0            East       DeliverIT                      600
1           North           FastX                      450
2           South       QuickShip                      200
3            West         ShipNow                      400


In [0]:
#Problem4-which orders are high-risk?
avg_penalty = df['penalty'].mean()
print(avg_penalty)
high_risk_orders = df[
    (df['delay_days'] > 0) &
    (df['penalty'] > avg_penalty)
]
hro=high_risk_orders[
    ['order_id', 'delivery_region', 'courier_partner',
     'delay_days', 'penalty', 'order_value']
]
print(hro)
hro.to_csv("high_risk_orders.csv",index=False)

206.25
   order_id delivery_region courier_partner  delay_days  penalty  order_value
2         3            East       DeliverIT           3      600         3000
4         5           North           FastX           2      300         4000
7         8            West         ShipNow           2      400         3500


In [0]:
#Problem5-Which warehouse is underperforming?
warehouse_perf = (
    df.groupby('warehouse_location')
      .agg(
          total_orders=('order_id', 'count'),
          delayed_orders=('is_delayed', 'sum'),
          total_penalty_by_region=('penalty', 'sum')
      )
)

warehouse_perf['delay_rate_%'] = (
    warehouse_perf['delayed_orders'] /
    warehouse_perf['total_orders']
) * 100
warehouse_perf=warehouse_perf.reset_index()
warehouse_perf.to_csv("warehouse_performance.csv",index=False)
warehouse_perf.sort_values(by='delay_rate_%', ascending=False)
#warehouse_perf.to_csv("warehouse_performance.csv",index=False)

,warehouse_location,total_orders,delayed_orders,total_penalty_by_region,delay_rate_%
1,Delhi_WH,2,2,450,100.0
0,Bangalore_WH,2,1,200,50.0
2,Kolkata_WH,2,1,600,50.0
3,Mumbai_WH,2,1,400,50.0


In [0]:
#Problem6-which combination of region + courier is worst?
combo_perf = (
    df.groupby(['delivery_region', 'courier_partner'])
      .agg(
          total_orders=('order_id', 'count'),
          delayed_orders=('is_delayed', 'sum'),
          avg_delay_days=('delay_days', 'mean')
      )
)

combo_perf['delay_rate_%'] = (
    combo_perf['delayed_orders'] /
    combo_perf['total_orders']
) * 100
combo_perf=combo_perf.reset_index()
combo_perf.to_csv("combination_performance.csv",index=False)
combo_perf.sort_values(by='delay_rate_%', ascending=False)

,delivery_region,courier_partner,total_orders,delayed_orders,avg_delay_days,delay_rate_%
1,North,FastX,2,2,1.5,100.0
0,East,DeliverIT,2,1,1.5,50.0
2,South,QuickShip,2,1,1.0,50.0
3,West,ShipNow,2,1,1.0,50.0


In [0]:
#Problem7-what is the average delay days per region?
avg_delay_by_region = (
    df.groupby('delivery_region').agg(avg_delay_days=('delay_days','mean'))
).reset_index()
print(avg_delay_by_region)
avg_delay_by_region.to_csv("average_delay_per_region.csv",index=False)

  delivery_region  avg_delay_days
0            East             1.5
1           North             1.5
2           South             1.0
3            West             1.0


In [0]:
#Extra Question-what percentage of orders are delayed?
delay_percentage = (df['delay_days'] > 0).mean() * 100
delay_percentage

np.float64(62.5)

In [0]:
#Problem8-which courier contributes most to penalty loss?
ms=df.groupby('courier_partner').agg(total_penalty_by_region=('penalty','sum')).reset_index()
print(ms)
ms.to_csv("most_penalty.csv",index=False)

  courier_partner  total_penalty_by_region
0       DeliverIT                      600
1           FastX                      450
2       QuickShip                      200
3         ShipNow                      400


In [0]:
#Problem9-which day has most delays?
df['is_delayed'] = df['delay_days'] > 0
df['order_day'] = df['order_date'].dt.day_name()

dmd=df[df['is_delayed']].groupby('order_day')['order_id'].count().reset_index(name='no_of_delay')
print(dmd)

dmd.to_csv("day_order_delay1.csv",index=False)

   order_day  no_of_delay
0     Friday            1
1     Monday            2
2   Saturday            1
3  Wednesday            1


In [0]:
df.to_csv("output.csv",index=False)